# فصل ۱۲: آشنایی با پایگاه داده‌های برداری (Vector Databases)
## بخش ۶: فیلترینگ مبتنی بر فراداده (Metadata Filtering)

تا اینجا تمام تمرکز ما بر روی **متن (Content)** و شباهت برداری آن بود. اما در دنیای واقعی داده‌ها دارای **زمینه (Context)** نیز هستند: سال انتشار، دپارتمان، نویسنده، برچسب‌ها و سطح دسترسی.

> **خطر توهم در بازیابی (Retrieval Hallucination):**
> فرض کنید اسناد مالی ۵ سال گذشته یک شرکت در دیتابیس ذخیره شده باشد. کاربر می‌پرسد: *«درآمد شرکت در سال ۲۰۲۴ چقدر بود؟»*
> از نظر معنایی، متن گزارش مالی ۲۰۲۰ («درآمد شرکت عالی بود و به سوددهی رسیدیم...») به شدت شبیه گزارش ۲۰۲۴ است. در صورت جستجوی صرفاً معنایی، ممکن است دیتابیس عدد سال ۲۰۲۰ را به عنوان مرتبط‌ترین پاسخ برگرداند و مدل زبانی یک عدد کاملاً اشتباه تحویل کاربر دهد!

**راه‌حل استاندارد:** فیلترینگ ترکیبی (Hybrid / Metadata Filtering)؛ یعنی **ابتدا فضای داده‌ها با شروط ساخت‌یافته محدود شود و سپس جستجوی معنایی انجام گیرد.**

### ۱. آماده‌سازی محیط و مدل تعبیه متن (Embedding Function)

برای بردارسازی از مدل چندزبانه و محلی **HuggingFace** (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) استفاده می‌کنیم.

In [1]:
import os
from dotenv import load_dotenv

# حذف پروکسی‌های غیرفعال محیطی
for var in ["http_proxy", "https_proxy", "HTTP_PROXY", "HTTPS_PROXY"]:
    os.environ.pop(var, None)

os.environ["HF_HUB_OFFLINE"] = "1"
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# راه‌اندازی مدل تعبیه متن محلی
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print("✅ Embedding Function Ready!")

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4074.31it/s]


✅ Embedding Function Ready!


### ۲. سناریوی عملی: تیکت‌های پشتیبانی سازمان همراه با متادیتای چندبعدی

مجموعه‌ای از ۵ تیکت پشتیبانی را همراه با اطلاعات بخش سازمانی (`dept`)، وضعیت تیکت (`status`) و اولویت رسیدگی (`priority`) ذخیره می‌کنیم:

In [2]:
tickets = [
    "سرعت اینترنت در طبقه دوم خیلی کند شده است.",
    "سرعت اینترنت در طبقه دوم عالی است و مشکلی نداریم.",
    "دستگاه قهوه‌ساز خراب شده و روشن نمی‌شود.",
    "نرم‌افزار حسابداری خطای لایسنس می‌دهد.",
    "سرور اصلی داون شده و سایت بالا نمی‌آید."
]

metadatas = [
    {"dept": "IT", "status": "open", "priority": "low"},
    {"dept": "IT", "status": "closed", "priority": "low"},       # تیکت بسته شده
    {"dept": "HR", "status": "open", "priority": "medium"},
    {"dept": "Finance", "status": "open", "priority": "high"},
    {"dept": "IT", "status": "open", "priority": "critical"},  # بحرانی
]

ids = [f"ticket_{i}" for i in range(1, len(tickets) + 1)]

# ساخت پایگاه برداری درون حافظه (یا پایدار)
vector_db = Chroma.from_texts(
    texts=tickets,
    metadatas=metadatas,
    ids=ids,
    embedding=embedding_function,
    collection_name="support_tickets_collection"
)

print(f"✅ تعداد تیکت‌های ثبت‌شده در پایگاه برداری: {vector_db._collection.count()}")

✅ تعداد تیکت‌های ثبت‌شده در پایگاه برداری: 5


### ۳. فیلترهای ساده تک‌شرطی (Simple Filtering)

اگر کاربری بپرسد: *«مشکل قطعی اینترنت و شبکه»*، بدون فیلتر ممکن است به دلیل کلمه «خراب»، تیکت قهوه‌ساز یا نرم‌افزار حسابداری هم در نتایج برتر ظاهر شود.

با اعمال آرگومان **`filter={"dept": "IT"}`**، دیتابیس منحصراً اسناد دپارتمان فناوری اطلاعات را بررسی می‌کند:

In [3]:
query = "مشکل قطعی اینترنت و شبکه"

# جستجوی معنایی منحصراً در تیکت‌های دپارتمان IT
results = vector_db.similarity_search(
    query,
    k=2,
    filter={"dept": "IT"}
)

print(f"🔍 نتایج جستجو برای '{query}' با فیلتر dept='IT':\n")
for idx, doc in enumerate(results, 1):
    print(f"نتیجه {idx}:")
    print(f"  متن: {doc.page_content}")
    print(f"  متادیتا: {doc.metadata}\n")

🔍 نتایج جستجو برای 'مشکل قطعی اینترنت و شبکه' با فیلتر dept='IT':

نتیجه 1:
  متن: سرعت اینترنت در طبقه دوم خیلی کند شده است.
  متادیتا: {'status': 'open', 'dept': 'IT', 'priority': 'low'}

نتیجه 2:
  متن: سرور اصلی داون شده و سایت بالا نمی‌آید.
  متادیتا: {'dept': 'IT', 'priority': 'critical', 'status': 'open'}



### ۴. عملگرهای پیشرفته به سبک مونگو دی‌بی (ChromaDB Filter Operators)

پایگاه داده ChromaDB برای پرس‌وجوهای پیچیده از سینتکسی مشابه با **MongoDB** با پیشوند `$` پشتیبانی می‌کند:

| عملگر | عنوان | مثال | توضیح |
| :--- | :--- | :--- | :--- |
| **`$eq`** | مساوی (Equal) | `{"status": {"$eq": "open"}}` | تطابق دقیق مقدار |
| **`$ne`** | نامساوی (Not Equal) | `{"dept": {"$ne": "HR"}}` | حذف یک مقدار خاص |
| **`$gt` / `$gte`** | بزرگتر / مساوی | `{"year": {"$gte": 2022}}` | مقایسه عددی یا تاریخ |
| **`$lt` / `$lte`** | کوچکتر / مساوی | `{"score": {"$lt": 50}}` | مقایسه مقادیر کوچکتر |
| **`$in`** | عضو لیست (In Array) | `{"priority": {"$in": ["high", "critical"]}}` | تطابق با هر کدام از گزینه‌ها |
| **`$nin`** | عدم عضویت | `{"status": {"$nin": ["closed", "archived"]}}` | رد کردن گزینه‌های مشخص |
| **`$and`** | ترکیب عطفی (Logical AND) | `{"$and": [cond1, cond2]}` | برقراری همزمان تمام شرط‌ها |
| **`$or`** | ترکیب فصلی (Logical OR) | `{"$or": [cond1, cond2]}` | برقراری حداقل یکی از شرط‌ها |

### ۵. پیاده‌سازی کوئری چندشرطی پیشرفته (Complex Filter)

فرض کنید مدیر فنی سازمان سوال می‌پرسد:
> *«چه مشکلات مهمی در بخش IT هنوز باز هستند؟»*

این پرسش در زبان متادیتا به شروط زیر تبدیل می‌شود:
1. دپارتمان (`dept`) باید `IT` باشد (`$eq`).
2. وضعیت (`status`) باید `open` باشد (`$eq`).
3. اولویت (`priority`) باید `high` یا `critical` باشد (`$in`).

In [4]:
complex_filter = {
    "$and": [
        {"dept": {"$eq": "IT"}},
        {"status": {"$eq": "open"}},
        {"priority": {"$in": ["high", "critical"]}}
    ]
}

print("--- اجرای جستجوی پیشرفته چندشرطی ---")
complex_results = vector_db.similarity_search(
    "مشکلات سیستمی و سرور",
    k=2,
    filter=complex_filter
)

print(f"تعداد اسناد منطبق بر تمام شروط: {len(complex_results)}\n")
for doc in complex_results:
    print(f"* متن تیکت: {doc.page_content}")
    print(f"  متادیتا:  {doc.metadata}")

--- اجرای جستجوی پیشرفته چندشرطی ---
تعداد اسناد منطبق بر تمام شروط: 1

* متن تیکت: سرور اصلی داون شده و سایت بالا نمی‌آید.
  متادیتا:  {'status': 'open', 'dept': 'IT', 'priority': 'critical'}


### ۶. مقایسه معماری Pre-filtering در برابر Post-filtering

```
[رویکرد اول: Post-filtering]
Query  --->  [جستجوی HNSW روی کل دیتابیس (Top 100)]  --->  [اعمال فیلتر متادیتا]
                                                            ⚠️ خطر: اگر شرط سختگیرانه باشد،
                                                            ممکن است تمام ۱۰۰ سند حذف شده
                                                            و خروجی کاملاً خالی بماند!

[رویکرد دوم: Pre-filtering]
Query  --->  [اعمال فیلتر روی Storage / SQLite]  --->  [جستجوی برداری درون زیرمجموعه فیلترشده]
                                                        ✅ مزیت: تضمین بازگشت k نتیجه (در صورت وجود)
                                                        و سرعت پردازش بالاتر.
```

- دیتابیس **ChromaDB** به صورت هوشمند از رویکرد **Pre-filtering** استفاده می‌کند تا هم سرعت جستجو بهینه بماند و هم تضمین شود نتایج برگردانده‌شده همواره معتبر و غیرخالی هستند.

### ۷. نگاه به آینده: مفهوم جستجوی خودکار (Self-Querying Retriever)

در سیستم‌های نهایی کاربران انسانی هرگز دیکشنری‌های پایتونی مثل `{"$and": ...}` را نمی‌نویسند! کاربر به زبان طبیعی می‌گوید:
> *«تیکت‌های بحرانی باز آی‌تی درباره قطعی سرور را نشان بده»*

تکنیک **Self-Querying** در LangChain این چالش را با یک معماری دو مرحله‌ای حل می‌کند:
1. یک مدل زبانی (LLM) با دریافت سوال کاربر، ساختار متادیتای جدول را می‌خواند و دو خروجی تولید می‌کند:
   - **Query معنایی:** *«قطعی سرور»*
   - **Filter ساخت‌یافته:** `{"$and": [{"dept": "IT"}, {"status": "open"}, {"priority": "critical"}]}`
2. سپس LangChain این دو پارامتر را به دیتابیس برداری (Chroma) ارسال کرده و دقیق‌ترین اسناد را استخراج می‌کند.

*(در فصول بعدی که به سیستم‌های **RAG** می‌پردازیم، این ساختار خودکار را به طور کامل پیاده‌سازی خواهیم کرد.)*

### 🎯 جمع‌بندی نهایی فصل ۱۲ (پایگاه داده‌های برداری):

در این فصل پایه‌های بنیادین کار با وکتور دیتابیس‌ها را با هم پیمودیم:
1. **چرا دیتابیس‌های سنتی شکست می‌خورند:** جستجوی کلمات کلیدی توانایی درک مترادف‌ها و مفاهیم معنایی را ندارد.
2. **معماری دو لایه:** تفاوت لایه انبار داده پایدار روی دیسک (Storage) و موتور ایندکس سریع در رم (HNSW Index).
3. **ریاضیات شباهت:** برتری شباهت کسینوسی در NLP و نحوه شبیه‌سازی آن با ضرب داخلی بردارهای نرمال‌شده.
4. **راه‌اندازی پایدار:** ذخیره‌سازی دائمی با `persist_directory` و مدیریت فایل‌های `chroma.sqlite3`.
5. **چرخه حیات داده (CRUD):** مدیریت شناسه‌ها برای پیشگیری از اسناد تکراری (Upsert) و متدهای `get` و `delete`.
6. **فیلترینگ متادیتا:** پیش‌فیلتر شروط ساخت‌یافته به سبک MongoDB برای از بین بردن خطای توهم بازیابی.

این مفاهیم سنگ بنای اصلی **سیستم‌های تولید مبتنی بر بازیابی (RAG)** هستند که در فصول بعدی مستقیماً به آن‌ها نیاز خواهیم داشت.